In [6]:
import pandas as pd
import os
import re

departure_dir = '/Users/jackzipper/QSS20/final_project/final_project_data/ocha_monthly_departures'
returnee_dir = '/Users/jackzipper/QSS20/final_project/final_project_data/ocha_monthly_returnees'

eastern_provinces = ['Nord-kivu', 'Sud-kivu', 'Ituri']

month_map = {
    'janvier': '01', 'fevrier': '02', 'fev': '02', 'ferv': '02',
    'mars': '03', 'avril': '04', 'mai': '05', 'juin': '06',
    'juillet': '07', 'aout': '08', 'septembre': '09',
    'octobre': '10', 'novembre': '11', 'decembre': '12'
}

usecols = ['id', 'movement_date', 'evaluation_date', 'household', 'person',
           'population_id', 'population_label', 'cause_label',
           'admin1_id', 'admin1_label', 'admin2_label',
           'admin1_id_from', 'admin1_label_from']

def extract_snapshot_month(filename):
    filename = filename.lower()
    year_match = re.search(r'(20\d{2})', filename)
    if not year_match:
        return None
    year = year_match.group(1)
    for month_fr, month_num in month_map.items():
        if month_fr in filename:
            return f"{year}-{month_num}"
    return None

def get_sheet_name(xl):
    valid_sheets = [
        'Data', 'Data2', 'DATA', 'Data (2)',
        'IDPs', 'Idps',
        'Retournées', 'Retournée', 'Retournee', 'Retournés', 'Retour',
        'Déplacées',
        'Feuil1', 'sheet',
        'Sheet1', 'Sheet2',
        'factsheet_mai_2023', 'factsheet_mai',
        'Facteesheet-janv2023', 'Factsheet janv 2023',
        'Factsheet-Octobre2021',
        '01_source'
    ]
    for name in valid_sheets:
        if name in xl.sheet_names:
            return name
    return None

def open_excel(filepath):
    """Try openpyxl first, fall back to xlrd for old-format files."""
    for engine in ['openpyxl', 'xlrd']:
        try:
            return pd.ExcelFile(filepath, engine=engine), engine
        except Exception:
            continue
    return None, None

def load_files(directory):
    all_dfs = []
    all_files = os.listdir(directory)

    files_to_skip = [
        f for f in all_files
        if not f.endswith('__1_.xlsx')
        and f.replace('.xlsx', '__1_.xlsx') in all_files
    ]

    for filename in sorted(all_files):
        if not filename.endswith('.xlsx'):
            continue
        if filename in files_to_skip:
            print(f"  Skipping (superseded): {filename}")
            continue

        snapshot_month = extract_snapshot_month(filename)
        if not snapshot_month:
            print(f"  Skipping (no date found): {filename}")
            continue

        filepath = os.path.join(directory, filename)

        xl, engine = open_excel(filepath)
        if xl is None:
            print(f"  Skipping (unreadable format): {filename}")
            continue

        sheet_name = get_sheet_name(xl)
        if not sheet_name:
            print(f"  Skipping (no valid sheet): {filename}")
            continue

        try:
            df = pd.read_excel(
                filepath,
                sheet_name=sheet_name,
                engine=engine,
                usecols=lambda x: x in usecols
            )
        except Exception as e:
            print(f"  Skipping (read error): {filename} | {e}")
            continue

        if 'admin1_label' not in df.columns or 'person' not in df.columns:
            print(f"  Skipping (missing required columns): {filename}")
            continue

        df['snapshot_month'] = snapshot_month
        df['source_file'] = filename
        all_dfs.append(df)
        print(f"  Loaded: {filename} -> {snapshot_month} ({len(df)} rows)")

    return pd.concat(all_dfs, ignore_index=True)

# Load departures
print("Loading departure files...")
df_departures = load_files(departure_dir)
print(f"\nTotal departure rows: {len(df_departures)}")

# Load returnees
print("\nLoading returnee files...")
df_returnees = load_files(returnee_dir)
print(f"\nTotal returnee rows: {len(df_returnees)}")

Loading departure files...
  Loaded: rdc_mouvement_de_population_deplace_aout_2021.xlsx -> 2021-08 (31008 rows)
  Loaded: rdc_mouvement_de_population_deplace_aout_2023.xlsx -> 2023-08 (43656 rows)
  Loaded: rdc_mouvement_de_population_deplace_avril_2023.xlsx -> 2023-04 (39221 rows)
  Loaded: rdc_mouvement_de_population_deplace_avril_2024.xlsx -> 2024-04 (47289 rows)
  Loaded: rdc_mouvement_de_population_deplace_avril_2025.xlsx -> 2025-04 (41383 rows)
  Loaded: rdc_mouvement_de_population_deplace_decembre_2023.xlsx -> 2023-12 (48321 rows)
  Loaded: rdc_mouvement_de_population_deplace_decembre_2024.xlsx -> 2024-12 (36707 rows)
  Loaded: rdc_mouvement_de_population_deplace_fev_2024.xlsx -> 2024-02 (48479 rows)
  Loaded: rdc_mouvement_de_population_deplace_fev_2025.xlsx -> 2025-02 (45279 rows)
  Loaded: rdc_mouvement_de_population_deplace_fevrier_2023.xlsx -> 2023-02 (37716 rows)
  Loaded: rdc_mouvement_de_population_deplace_janvier_2023.xlsx -> 2023-01 (38230 rows)
  Loaded: rdc_mouvement

In [10]:
# Save departees and returnees as separate CSVs
df_departures.to_csv('/Users/jackzipper/QSS20/final_project/final_project_data/departees_eastern_drc.csv', index=False)
df_returnees.to_csv('/Users/jackzipper/QSS20/final_project/final_project_data/returnees_eastern_drc.csv', index=False)

print(f"Departees saved: {len(df_departures):,} rows")
print(f"Returnees saved: {len(df_returnees):,} rows")

Departees saved: 1,788,717 rows
Returnees saved: 459,202 rows


In [11]:
# Convert movement_date
df_departures['movement_date'] = pd.to_datetime(df_departures['movement_date'], errors='coerce')
df_returnees['movement_date'] = pd.to_datetime(df_returnees['movement_date'], errors='coerce')
df_departures['snapshot_month_dt'] = pd.to_datetime(df_departures['snapshot_month'])
df_returnees['snapshot_month_dt'] = pd.to_datetime(df_returnees['snapshot_month'])

eastern_provinces = ['Nord-kivu', 'Sud-kivu', 'Ituri']

# Only keep rows where movement_date falls within that snapshot month
df_departures_monthly = df_departures[
    (df_departures['admin1_label'].isin(eastern_provinces)) &
    (df_departures['movement_date'].dt.year == df_departures['snapshot_month_dt'].dt.year) &
    (df_departures['movement_date'].dt.month == df_departures['snapshot_month_dt'].dt.month)
]

df_returnees_monthly = df_returnees[
    (df_returnees['admin1_label'].isin(eastern_provinces)) &
    (df_returnees['movement_date'].dt.year == df_returnees['snapshot_month_dt'].dt.year) &
    (df_returnees['movement_date'].dt.month == df_returnees['snapshot_month_dt'].dt.month)
]

# Deduplicate on id alone - each event should only be counted once
# Sort by snapshot_month first so we keep the earliest appearance
df_departures_monthly = df_departures_monthly.sort_values('snapshot_month').drop_duplicates(subset=['id'], keep='first')
df_returnees_monthly = df_returnees_monthly.sort_values('snapshot_month').drop_duplicates(subset=['id'], keep='first')

print(f"Departures after dedup: {len(df_departures_monthly):,}")
print(f"Returnees after dedup: {len(df_returnees_monthly):,}")

# Aggregate to province-month
df_dep_panel = (df_departures_monthly
    .groupby(['admin1_label', 'snapshot_month'], as_index=False)
    .agg(
        total_displaced=('person', 'sum'),
        num_sites_displaced=('id', 'nunique')
    )
    .sort_values(['admin1_label', 'snapshot_month'])
)

df_ret_panel = (df_returnees_monthly
    .groupby(['admin1_label', 'snapshot_month'], as_index=False)
    .agg(
        total_returnees=('person', 'sum'),
        num_sites_returnees=('id', 'nunique')
    )
    .sort_values(['admin1_label', 'snapshot_month'])
)

# Merge and calculate net flow
df_panel = pd.merge(
    df_dep_panel,
    df_ret_panel,
    on=['admin1_label', 'snapshot_month'],
    how='outer'
).fillna(0).sort_values(['admin1_label', 'snapshot_month'])

df_panel['net_monthly_flow'] = df_panel['total_displaced'] - df_panel['total_returnees']

print(f"\nFinal panel rows: {len(df_panel)}")
print(f"\nTimepoints per province:")
print(df_panel.groupby('admin1_label')['snapshot_month'].count().sort_values(ascending=False).to_string())
print(f"\nDate range per province:")
print(df_panel.groupby('admin1_label')['snapshot_month'].agg(['min', 'max']).to_string())
print(f"\nSample - Nord-kivu:")
print(df_panel[df_panel['admin1_label'] == 'Nord-kivu']
      [['snapshot_month', 'total_displaced', 'total_returnees', 'net_monthly_flow']]
      .to_string())



Departures after dedup: 5,101
Returnees after dedup: 2,918

Final panel rows: 57

Timepoints per province:
admin1_label
Nord-kivu    24
Ituri        18
Sud-kivu     15

Date range per province:
                  min      max
admin1_label                  
Ituri         2021-09  2026-01
Nord-kivu     2021-09  2026-02
Sud-kivu      2021-10  2026-02

Sample - Nord-kivu:
   snapshot_month  total_displaced  total_returnees  net_monthly_flow
18        2021-09           2205.0          25156.0          -22951.0
19        2022-03              0.0          10890.0          -10890.0
20        2022-05          51489.0          32840.0           18649.0
21        2022-06           2692.0              0.0            2692.0
22        2022-10         166053.0              0.0          166053.0
23        2023-01          28684.0              0.0           28684.0
24        2023-03           7748.0          15299.0           -7551.0
25        2023-05          24344.0          14031.0           10313.0


In [50]:
# Save
df_panel.to_csv('/Users/jackzipper/QSS20/final_project_data/idp_dat_eastern_drc.csv', index=False)